# HybridFER-4E — ViT-B/16 단일 (fallback, val 0.8458)

4클래스 얼굴 감정: anger / happy / panic / sadness.

TF 없이 torch 만 있으면 돌아감. 앙상블 대비 성능은 3%p 낮지만 파일 1개 (328MB), cold 12초라 가벼움.

- Repo: https://github.com/moneyally/yua-encoder
- Release: https://github.com/moneyally/yua-encoder/releases/tag/v1.0.0
- Checkpoint: `exp05_vit_b16_two_stage.pt`

## Cell 1. 환경 체크

In [ ]:
import sys, os, platform
IS_COLAB = 'google.colab' in sys.modules
print(f'python {platform.python_version()}  colab={IS_COLAB}')
if IS_COLAB:
    import subprocess
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
    print(f'gpu    {r.stdout.strip() or "GPU 없음 → Runtime > GPU 로 변경"}')
else:
    try:
        import torch
        print(f'torch  {torch.__version__}  cuda={torch.cuda.is_available()}')
    except ImportError:
        print('torch 없음. Cell 3 에서 설치.')

## Cell 2. Repo 확보

In [ ]:
from pathlib import Path
if IS_COLAB:
    REPO = Path('/content/yua-encoder')
    if not REPO.is_dir():
        !git clone --depth 1 https://github.com/moneyally/yua-encoder.git {REPO}
    os.chdir(REPO)
else:
    REPO = Path.cwd()
    while not (REPO / 'predict.py').is_file() and REPO.parent != REPO:
        REPO = REPO.parent
    os.chdir(REPO)
print(f'repo   {REPO}')
assert (REPO / 'predict.py').is_file(), 'predict.py 없음'

## Cell 3. 의존성 설치 (Colab 전용, TF 제외)

torch 는 Colab 기본. timm + facenet-pytorch 만.

In [ ]:
if IS_COLAB:
    !pip install -q \
        'timm==1.0.26' \
        'facenet-pytorch==2.6.0' \
        'Pillow' 'scikit-learn'
    print('install done (TF 없이 torch-only)')
else:
    print('로컬 — 기존 env 사용.')

## Cell 4. ViT 체크포인트 (328MB)

SHA256 검증 포함.

In [ ]:
import hashlib, urllib.request
MODELS_DIR = REPO / 'models'
MODELS_DIR.mkdir(exist_ok=True)
BASE = 'https://github.com/moneyally/yua-encoder/releases/download/v1.0.0'

sums_path = REPO / 'SHA256SUMS.txt'
if not sums_path.is_file():
    urllib.request.urlretrieve(f'{BASE}/SHA256SUMS.txt', sums_path)
expected = {}
with open(sums_path) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        h, p = line.split(None, 1)
        expected[Path(p).name] = h

name = 'exp05_vit_b16_two_stage.pt'
out = MODELS_DIR / name

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(1 << 20)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

if out.is_file() and expected.get(name) == sha256_of(out):
    print(f'[cached] {name}')
else:
    print(f'[dl] {name} (328MB)', end=' ', flush=True)
    urllib.request.urlretrieve(f'{BASE}/{name}', out)
    got = sha256_of(out)
    ok = (expected.get(name) == got)
    print('ok' if ok else 'hash mismatch')
    assert ok, f'{name} SHA256 불일치'
print(f'ready  {out}')

## Cell 5. 모델 로드

torch 만. XLA compile 없음 → cold 비용 작음 (12초 내).

In [ ]:
import time, sys
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from predict import load_model, predict, predict_probs, CLASSES

print(f'classes {CLASSES}')
t0 = time.perf_counter()
model = load_model(str(MODELS_DIR / 'exp05_vit_b16_two_stage.pt'),
                   tta=False, auto_face_crop=True)
print(f'load {time.perf_counter() - t0:.1f}s  type={model["type"]}')

## Cell 6. 1장 추론

In [ ]:
IMAGE_PATH = None  # 자기 이미지 경로 (예: '/content/my_face.jpg')

if IMAGE_PATH is None:
    val_dir = REPO / 'data_rot/img/val'
    if val_dir.is_dir():
        for cls_dir in sorted(val_dir.iterdir()):
            for ip in sorted(cls_dir.glob('*.jpg'))[:1]:
                IMAGE_PATH = str(ip)
                break
            if IMAGE_PATH:
                break

if IMAGE_PATH is None and IS_COLAB:
    print('파일 올려. 얼굴 사진 1장.')
    from google.colab import files
    up = files.upload()
    if up:
        IMAGE_PATH = '/content/' + list(up.keys())[0]

if IMAGE_PATH is None:
    print('IMAGE_PATH 미지정. 맨 위 변수 직접 수정하거나 업로드 필요.')
else:
    print(f'input  {IMAGE_PATH}')
    _ = predict(model, IMAGE_PATH)  # warmup (가벼움)
    t0 = time.perf_counter()
    label, conf = predict(model, IMAGE_PATH)
    dt = time.perf_counter() - t0
    probs = predict_probs(model, IMAGE_PATH)
    print(f'pred   {label}  conf={conf:.3f}  warm={dt*1000:.0f}ms')
    for c, p in zip(CLASSES, probs):
        bar = '█' * int(p * 40)
        print(f'  {c:8s} {p:.3f}  {bar}')

## Cell 7. val 20장 정확도 (옵션)

In [ ]:
import random
import numpy as np

val_dir = REPO / 'data_rot/img/val'
if not val_dir.is_dir():
    print('val 폴더 없음. skip.')
else:
    rng = random.Random(42)
    samples = []
    for cls in CLASSES:
        imgs = sorted((val_dir / cls).glob('*.jpg'))
        samples += [(str(p), cls) for p in rng.sample(imgs, 5)]
    y_true, y_pred = [], []
    for path, gt in samples:
        p = predict_probs(model, path)
        y_true.append(CLASSES.index(gt))
        y_pred.append(int(np.argmax(p)))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    acc = float((y_true == y_pred).mean())
    print(f'acc (n=20) {acc:.3f}')
    print('confusion matrix (row=gt, col=pred)')
    print('        ' + '  '.join(f'{c:>7s}' for c in CLASSES))
    cm = np.zeros((4, 4), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    for i, c in enumerate(CLASSES):
        print(f'{c:>7s} ' + '  '.join(f'{v:>7d}' for v in cm[i]))

---

val 1200 기준 accuracy 0.8458. Warm p95 0.68s. 성능 더 원하면 메인 앙상블 (val 0.8692) 쪽 (`01_ensemble_colab.ipynb`).